In [ ]:
                                               # Chapter 5. Text Clustering and Topic Modeling

In [ ]:
# install these things
!pip install -U \
transformers==4.46.3 \
huggingface_hub==0.26.2 \
accelerate==1.1.1 \
tokenizers==0.20.3 \
safetensors

In [ ]:
# Load data from Hugging Face
from datasets import load_dataset

# Download and load the "arxiv_nlp" training dataset
dataset = load_dataset("maartengr/arxiv_nlp")["train"]

# Get all the abstracts and titles from the dataset
abstracts = dataset["Abstracts"]
titles = dataset["Titles"]

In [ ]:
# A Common Pipeline for Text Clustering
# Embedding Documents

In [ ]:
# Import the SentenceTransformer class
from sentence_transformers import SentenceTransformer

# Load the pre-trained GTE-Small embedding model
embedding_model = SentenceTransformer("thenlper/gte-small")

# Convert all abstracts into numerical vector embeddings
embeddings = embedding_model.encode(abstracts,
show_progress_bar=True    # Show the embedding generation progress
)

print(embeddings.shape)

In [ ]:
# Reducing the Dimensionality of Embeddings

In [ ]:
# Import the UMAP class for dimensionality reduction
from umap import UMAP

# We reduce the input embeddings from 384 dimensions to 5 dimensions
umap_model = UMAP(
    n_components=5,   # Reduce the embeddings to 5 dimensions
    min_dist=0.0,     # Keep similar data points close together
    metric='cosine',  # Use cosine distance to measure similarity
    random_state=42   # Set a fixed random seed for reproducible results
)

# Reduce the dimensions of the embeddings
reduced_embeddings = umap_model.fit_transform(embeddings)
print(reduced_embeddings)

In [ ]:
# Cluster the Reduced Embeddings

In [ ]:
# Import the HDBSCAN class for clustering
from hdbscan import HDBSCAN


# Create and train the HDBSCAN model to group similar embeddings into clusters
hdbscan_model = HDBSCAN(
    min_cluster_size=50,    # Minimum number of documents needed to form a cluster

    metric="euclidean",      # Use Euclidean distance to measure similarity

    cluster_selection_method="eom"   # Use the EOM method to choose clusters
).fit(reduced_embeddings)

# Get the cluster label assigned to each document
clusters = hdbscan_model.labels_

print(clusters)

# How many clusters did we generate?
print(len(set(clusters)))

In [ ]:
#  Convert the abstracts from the dataset format to a Python list
print(type(abstracts))

In [ ]:
#A Hugging Face Column does not accept numpy.int64 indices.
# It only accepts normal Python int.
# use int(index)
# ==== OR ===
# Convert the entire column to a list once
# abstracts = list(abstracts)
# Then use the original code

In [ ]:
# Inspecting the Clusters

In [8]:
# Convert the abstracts into a Python list so they can be accessed by index.
abstracts = list(abstracts)

In [ ]:
import numpy as np

# Select the cluster number to view
cluster = 0

# Find the first 3 abstracts that belong to the selected cluster
for index in np.where(clusters == cluster)[0][:3]:

  # Print the first 300 characters of each abstract
    print(abstracts[index][:300] + "... \n")

In [ ]:
import pandas as pd
# Reduce 384-dimensional embeddings to two dimensions for easier visualization
reduced_embeddings = UMAP(
    n_components=2,      # Reduce to 2 dimension

    min_dist=0.0,        # Keep similar points close together

    metric="cosine",     # Use cosine distance

    random_state=42      # Set a fixed random seed

).fit_transform(embeddings)


# Create a DataFrame with the reduced embeddings
df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])

# Add the document titles
df["title"] = titles

# Add the cluster labels as strings
df["cluster"] = [str(c) for c in clusters]

# Select only the documents that belong to clusters
clusters_df = df.loc[df.cluster != "-1", :]

# Select only the outlier documents
outliers_df = df.loc[df.cluster == "-1", :]

print(clusters_df)
print(outliers_df)

In [ ]:
# Import the Matplotlib library for data visualization
import matplotlib.pyplot as plt


# Plot outliers and non-outliers separately
plt.scatter(outliers_df.x, outliers_df.y,
            alpha=0.05,      # Make the points transparent

            s=2,             # Set a small point size

           c="grey")          # Use grey color for outliers

# Plot the clustered documents with different colors for each cluster
plt.scatter(
clusters_df.x, clusters_df.y,
c=clusters_df.cluster.astype(int),   # Color points based on cluster labels
             alpha=0.6, s=2,
             cmap="tab20b"        # Use the Tab20b color map
)


# Hide the x-axis and y-axis

plt.axis("off")

In [ ]:
#                      From Text Clustering to Topic Modeling

In [ ]:
#  BERTopic

In [ ]:
# Install BERTopic and the required libraries for topic modeling and text embeddings
!pip install bertopic umap-learn hdbscan sentence-transformers

In [ ]:
# Import the BERTopic class
from bertopic import BERTopic

# Train our model with our previously defined models
topic_model = BERTopic(
    embedding_model=embedding_model,   # Use the embedding model to generate document embeddings

    umap_model=umap_model,             # Use UMAP to reduce embedding dimensions

    hdbscan_model=hdbscan_model,       # Use HDBSCAN to group similar documents into topics

    verbose=True)                      # Show the training progress

      # Input text documents
      # Precomputed embeddings for the documents
.fit(abstracts, embeddings)

In [ ]:
# Display information about all the topics discovered by the BERTopic model
print(topic_model.get_topic_info())

In [ ]:
# Display the keywords and their scores for Topic 0
print(topic_model.get_topic(0))

#  Find topics that are most similar to the given search text
print(topic_model.find_topics("topic modeling"))

# Display the keywords and their scores for Topic 22
print(topic_model.get_topic(22))

# Display the topic assigned to the specified document title
print(topic_model.topics_[titles.index("BERTopic: Neural topic modeling with a class-based TF-IDF procedure")])

In [ ]:
# Convert the titles into a Python list
titles=list(titles)

# Visualize the documents and their topics in a 2D plot
fig = topic_model.visualize_documents(
    titles,                             # Document titles to display

reduced_embeddings=reduced_embeddings,  # 2D embeddings for visualization

width=1200,                             # Set the width of the plot

hide_annotations=True)                  # Hide document labels for a cleaner view

# Update fonts of legend for easier visualization
fig.update_layout(font=dict(size=16))

In [ ]:
# Visualize barchart with ranked keywords
topic_model.visualize_barchart()

In [ ]:
# Visualize relationships between topics
topic_model.visualize_heatmap(n_clusters=30)

In [ ]:
# Visualize the potential hierarchical structure of topics
topic_model.visualize_hierarchy()

In [ ]:
#               Adding a Special Lego Block

In [ ]:
import pandas as pd
from copy import deepcopy

# Save original topic representations
original_topics = deepcopy(topic_model.topic_representations_)

def topic_differences(model, original_topics, nr_topics=5):
    """Show the differences in topic representations between two models."""

    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])

    for topic in range(nr_topics):
        # Extract top 5 words per topic from the original model
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])

        # Extract top 5 words per topic from the updated model
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])

        # Add a new row to the DataFrame
        df.loc[len(df)] = [topic, og_words, new_words]

    return df
topic_differences(topic_model, original_topics)

In [ ]:
#               KeyBERTInspired

In [ ]:
# Import the KeyBERTInspired representation model
from bertopic.representation import KeyBERTInspired

# Create a KeyBERTInspired model to generate better topic representations
representation_model = KeyBERTInspired()

# Update the topic representations using the KeyBERTInspired model
topic_model.update_topics(abstracts,    # inputs
representation_model=representation_model)


# Compare the updated topics with the original topics
topic_differences(topic_model, original_topics)

In [ ]:
#            Maximal marginal relevance

In [ ]:
# Import the Maximal Marginal Relevance representation model
from bertopic.representation import MaximalMarginalRelevance

# Create a Maximal Marginal Relevance model to generate more diverse topic keywords

representation_model = MaximalMarginalRelevance(diversity=0.2)    # Set the diversity level for selecting keywords

# Update the topic representations using the MMR model
topic_model.update_topics(abstracts,
representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

In [ ]:
           # The Text Generation Lego Block

In [ ]:
# Import the Hugging Face pipeline and BERTopic TextGeneration representation model
from transformers import pipeline
from bertopic.representation import TextGeneration

prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords:
'[KEYWORDS]'.

Based on the documents and keywords, what is this topic about?
"""

# Load Flan-T5
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

# Create representation model
representation_model = TextGeneration(
    generator,
    prompt=prompt,
    doc_length=50,
    tokenizer="whitespace"   # Split text using whitespace
)

# Update topic representations
topic_model.update_topics(
    abstracts,
    representation_model=representation_model
)

# Compare topics
topic_differences(topic_model, original_topics)

In [ ]:
# Install or update BERTopic and its required libraries
!pip install -U bertopic sentence-transformers umap-learn hdbscan

In [ ]:
            # Unable to do in openai . it needs billing

In [ ]:
import openai
from bertopic.representation import OpenAI
prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]
Based on the information above, extract a short topic label in
the following format:
topic: <short topic label>
"""
# Update our topic representations using GPT-3.5
client = openai.OpenAI(api_key="API_KEY")
representation_model = OpenAI(
    client, model="gpt-3.5-turbo", exponential_backoff=True,
chat=True, prompt=prompt
)
topic_model.update_topics(abstracts,
representation_model=representation_model)
# Show topic differences
topic_differences(topic_model, original_topics)


In [ ]:
# Visualize topics and documents
fig = topic_model.visualize_document_datamap(
    titles,
    topics=list(range(20)),
    reduced_embeddings=reduced_embeddings,
    width=1200,
    label_font_size=11,
    label_wrap_width=20,
    use_medoids=True,
)
